## Calibration for the stereo camera

- To do: Needs, thresholding to make image white-black instead of gray
- v1.1: Detected most checkerboards but still horrible result
- v1.0: Detected multiple checkerboards

In [ ]:
## 
import cv2
import glob
import numpy as np

def find_all_checkerboards_with_masks_enhanced(image, checkerboards_dims):
    detected_corners = []
    mask = np.ones_like(image, dtype=np.uint8)  # Initialize mask

    for checkerboard_dims in checkerboards_dims:
        while True:
            masked_image = cv2.bitwise_and(image, image, mask=mask)

            # Pre-process the image to improve contrast
            enhanced_image = cv2.equalizeHist(masked_image)

            # Find checkerboards with improved flags
            ret, corners = cv2.findChessboardCorners(enhanced_image, checkerboard_dims, 
                                                     flags=cv2.CALIB_CB_ADAPTIVE_THRESH + 
                                                           cv2.CALIB_CB_NORMALIZE_IMAGE)

            if ret:
                # Refine corners for accuracy
                # criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
                # corners = cv2.cornerSubPix(enhanced_image, corners, (11, 11), (-1, -1), criteria)
                
                detected_corners.append((corners, checkerboard_dims))

                # Mask out the detected checkerboard region (use convex hull for better masking)
                polygon_points = np.array([corner[0] for corner in corners], dtype=np.int32)
                cv2.fillConvexPoly(mask, polygon_points, 0)
            else:
                break

    return detected_corners

def processStereoImages(left_image_path, right_image_path, checkerboards_dims: list, visualize=True):
    left_img = cv2.imread(left_image_path)
    right_img = cv2.imread(right_image_path)
    gray_left = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)

    all_corners_left = find_all_checkerboards_with_masks_enhanced(gray_left, checkerboards_dims)
    all_corners_right = find_all_checkerboards_with_masks_enhanced(gray_right, checkerboards_dims)

    if visualize:
        for corners, dims in all_corners_left:
            cv2.drawChessboardCorners(left_img, dims, corners, True)
        for corners, dims in all_corners_right:
            cv2.drawChessboardCorners(right_img, dims, corners, True)

        cv2.imshow("Left Image with All Corners", left_img)
        cv2.imshow("Right Image with All Corners", right_img)
        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cv2.destroyAllWindows()

    return left_img, right_img, all_corners_left, all_corners_right

def stereoCalibration(left_img_dir, right_img_dir, corners, visualize=True):
    left_image_files = glob.glob(left_img_dir + r"\*.png")
    right_image_files = glob.glob(right_img_dir + r"\*.png")

    # left_image_files = [left_image_files[-1]]
    # right_image_files = [right_image_files[-1]]

    left_images = []
    right_images = []
    corners_left = []
    corners_right = []

    for left_image_file, right_image_file in zip(left_image_files, right_image_files):
        l, r, cl, cr = processStereoImages(left_image_file, right_image_file, corners, visualize=True)
        left_images.append(l)
        right_images.append(r)
        corners_left.append(cl)
        corners_right.append(cr)

    if visualize:
        left_image = left_images[-1]
        cs_l = corners_left[-1]
        right_image = right_images[-1]
        cs_r = corners_right[-1]
        for c_l, dims_l in cs_l:
            cv2.drawChessboardCorners(left_image, dims_l, c_l, True)
        for c_r, dims_r in cs_r:
            cv2.drawChessboardCorners(right_image, dims_r, c_r, True)

        cv2.imshow("Final Left Image with All Corners", left_image)
        cv2.imshow("Final Right Image with All Corners", right_image)

        while True:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cv2.destroyAllWindows()

# Example usage:
left_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_02\data"
right_img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\calib\image_03\data"

corners = [
    (7,11), (7,11), 
    (5,7), (5,7), (5,7), (5,7), (5,7), 
    (7,5), (7,5), (7,5), (7,5), (7,5), 
    (15, 5)
]
# corners = [
#     (11, 7), (11, 7),
#     (7, 5), (7, 5), (7, 5), (7, 5), (7, 5),
#     (5, 7), (5, 7), (5, 7), (5, 7), (5, 7), 
#     (5, 15)
# ]
stereoCalibration(left_img_path, right_img_path, corners, visualize=True)


## Kitti dataset LSVM
- have to have the structure:
```plaintext
kitti_dataset/
├── data_object_image_2/        # Left camera images
│   ├── training/
│   │   ├── image_2/            # Left camera images
│   │   └── image_3/            # Right camera images (optional if downloaded)
├── data_object_label_2/        # Labels
│   ├── training/
│   │   └── label_2/            # 3D object labels
├── data_object_calib/          # Calibration files
│   ├── training/
│   │   └── calib/              # Calibration data
```
-

In [ ]:
import pykitty
import cv2
import os
import numpy as np
from scipy.io import loadmat

In [ ]:

# Define paths
left_image_path = 'kitti_dataset/images/training/image_2'
right_image_path = 'kitti_dataset/images/training/image_3'

# Load an example frame (e.g., 000000.png)
frame_id = '000000'

left_image = cv2.imread(os.path.join(left_image_path, frame_id + '.png'))
right_image = cv2.imread(os.path.join(right_image_path, frame_id + '.png'))

# Display images
cv2.imshow('Left Image', left_image)
cv2.imshow('Right Image', right_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
label_path = 'kitti_dataset/labels/training/label_2'

def read_labels(label_file):
    labels = []
    with open(label_file, 'r') as file:
        for line in file:
            parts = line.strip().split(' ')
            obj = {
                'type': parts[0],  # Car, Pedestrian, Cyclist
                'bbox': [float(p) for p in parts[4:8]],  # 2D bounding box in image (xmin, ymin, xmax, ymax)
                'dimensions': [float(p) for p in parts[8:11]],  # 3D dimensions (h, w, l)
                'location': [float(p) for p in parts[11:14]],  # 3D location (x, y, z)
                'rotation_y': float(parts[14]),  # Rotation around Y-axis
            }
            labels.append(obj)
    return labels

# Example usage
frame_label_file = os.path.join(label_path, frame_id + '.txt')
labels = read_labels(frame_label_file)
print(labels)


In [ ]:
calib_path = 'kitti_dataset/calib/training/calib'

def read_calib(calib_file):
    with open(calib_file, 'r') as file:
        lines = file.readlines()
        calib = {}
        for line in lines:
            key, value = line.split(':', 1)
            calib[key] = np.array([float(x) for x in value.strip().split()])
        return calib

# Example usage
frame_calib_file = os.path.join(calib_path, frame_id + '.txt')
calib = read_calib(frame_calib_file)
print(calib['P2'])  # Projection matrix for left camera


In [4]:
import importlib
import detect2  # Your script
importlib.reload(detect2)

<module 'detect2' from 'C:\\Users\\szakt\\Desktop\\DTU\\Perception\\FinalProject\\yolov7\\detect2.py'>

In [ ]:
import sys
sys.path.append(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7")
import torch
from detect2 import main
import matplotlib.pyplot as plt
import cv2
img_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_rect\seq_01\image_02\data"
model_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\yolov7.pt"
output_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\Output"
main(img_path, model_path)

C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\models\experimental.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(w, map_location=map_loc

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


In [ ]:
import sys
import os
import torch
import cv2
import numpy as np
from pathlib import Path
import sys
sys.path.append(r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7")
from models.yolo import Model  # Import the YOLO model
import torchvision
from torch.nn import Module

# Helper functions
def xywh2xyxy(x):
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[:, 0] = x[:, 0] - x[:, 2] / 2  # top left x
    y[:, 1] = x[:, 1] - x[:, 3] / 2  # top left y
    y[:, 2] = x[:, 0] + x[:, 2] / 2  # bottom right x
    y[:, 3] = x[:, 1] + x[:, 3] / 2  # bottom right y
    return y

def clip_coords(boxes, img_shape):
    boxes[:, 0].clamp_(0, img_shape[1])  # x1
    boxes[:, 1].clamp_(0, img_shape[0])  # y1
    boxes[:, 2].clamp_(0, img_shape[1])  # x2
    boxes[:, 3].clamp_(0, img_shape[0])  # y2

def scale_coords(img1_shape, coords, img0_shape, ratio_pad=None):
    if ratio_pad is None:  # calculate from img0_shape
        gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])  # gain  = old / new
        pad = (img1_shape[1] - img0_shape[1] * gain) / 2, (img1_shape[0] - img0_shape[0] * gain) / 2  # wh padding
    else:
        gain = ratio_pad[0][0]
        pad = ratio_pad[1]

    coords[:, [0, 2]] -= pad[0]  # x padding
    coords[:, [1, 3]] -= pad[1]  # y padding
    coords[:, :4] /= gain
    clip_coords(coords, img0_shape)
    return coords

def non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45, classes=None):
    nc = prediction.shape[2] - 5  # number of classes
    xc = prediction[..., 4] > conf_thres  # candidates

    output = [torch.zeros((0, 6), device=prediction.device)] * prediction.shape[0]
    for xi, x in enumerate(prediction):
        x = x[xc[xi]]  # confidence
        if not x.shape[0]:
            continue

        x[:, 5:] *= x[:, 4:5]  # conf = obj_conf * cls_conf
        box = xywh2xyxy(x[:, :4])
        conf, j = x[:, 5:].max(1, keepdim=True)
        x = torch.cat((box, conf, j.float()), 1)[conf.view(-1) > conf_thres]

        if classes is not None:
            x = x[(x[:, 5:6] == torch.tensor(classes, device=x.device)).any(1)]

        n = x.shape[0]
        if not n:
            continue

        boxes, scores = x[:, :4], x[:, 4]
        i = torchvision.ops.nms(boxes, scores, iou_thres)
        output[xi] = x[i]
    return output

# Load YOLO model
yaml_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\cfg\deploy\yolov7.yaml"
weights_path = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\yolov7\yolov7.pt"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model(cfg=yaml_path).to(device)
checkpoint = torch.load(weights_path, map_location=device)
model.load_state_dict(checkpoint['model'].state_dict(), strict=False)
model.eval()

# Folder with image sequence
image_folder = r"C:\Users\szakt\Desktop\DTU\Perception\FinalProject\34759_final_project_raw\seq_01\image_02\data"
image_files = sorted([f for f in os.listdir(image_folder) if f.endswith(('.png', '.jpg', '.jpeg'))])

# Iterate through images
for img_file in image_files:
    img_path = os.path.join(image_folder, img_file)
    img0 = cv2.imread(img_path)
    img_resized = cv2.resize(img0, (640, 640))
    img_normalized = img_resized / 255.0
    img = torch.from_numpy(img_normalized).float().permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        predictions = model(img)[0]
        detections = non_max_suppression(predictions, conf_thres=0.25, iou_thres=0.45, classes=[0, 1, 2])

    for det in detections:
        if len(det):
            det[:, :4] = scale_coords(img.shape[2:], det[:, :4], img0.shape).round()
            for *xyxy, conf, cls in reversed(det):
                c1, c2 = (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3]))
                color = (0, 255, 0)
                cv2.rectangle(img0, c1, c2, color, 2)
                label = f'{int(cls)} {conf:.2f}'
                cv2.putText(img0, label, (c1[0], c1[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    cv2.imshow('Detections', img0)
    key = cv2.waitKey(0)  # Press any key to go to the next image
    if key == 27:  # Exit if 'Esc' is pressed
        break

cv2.destroyAllWindows()
